# 05 Kerala 2018 floods: descriptive replay

Question (T142 only): what did the 2018 monsoon look like in Kerala, from the records we hold,
and which of those records were public at the time? This is a **descriptive replay**, not a
forecast and not a model test (D29, D125). No historical forecast vintage for 2018 is held, so
none is shown or imitated.

Every replay record carries a mode (D127). **as_known**: the publisher's own dated issue exists
and falls on or before 30 Sep 2018; in the as-known view the record appears only from that date.
**reconstructed**: everything else, here the IMD gridded analysis (S08), which is a later
gauge-based reconstruction fetched in 2026. A fetch date is never a publication date.

Sources. S08 IMD 0.25 degree daily rain (local only: IMD forbids reproduction, so only derived
box means are exported), Kerala cell box 8.25-12.75 N, 74.75-77.5 E: it also holds Tamil Nadu
and Karnataka border cells, so it is a box, not the Kerala boundary. Climatology 1981-2010.
S36 KSDMA memoranda: NRSC flooded area as on 16 Jul 2018 and district fatalities for each
memorandum period (period totals, never spread over days). S32 KSEB dam levels: the stored
15 Aug 2018 report is parsed; it holds no dam row (the earliest report with rows is 26 Oct 2023).

Run from the repository root in a fresh Python 3.12 process. Outputs go to
`reports/eda/05_kl_flood2018/` (or `ATLAS_EDA_OUT`).

In [ ]:
import csv
import json
import os
import textwrap
from datetime import date, timedelta
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np

from atlas.analysis.quality import caption
from atlas.analysis.replay import LABEL, daily_box_mean, mode, validate
from atlas.config import load_state
from atlas.contracts import RawArtifact
from atlas.features.m05 import year_artifacts
from atlas.ingest.raw_store import RawStore
from atlas.ingest.s08 import BOXES, S08
from atlas.ingest.s32 import S32
from atlas.ingest.s32 import URL as S32_URL
from atlas.ingest.s36 import S36, TABLES

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/05_kl_flood2018"))
OUT.mkdir(parents=True, exist_ok=True)
RAW = Path("data/raw")
START, END = date(2018, 6, 1), date(2018, 9, 30)
BASE = range(1981, 2011)
DAYS = [START + timedelta(i) for i in range((END - START).days + 1)]
kl = load_state(Path("configs/states/kl.yaml"))


def stored(url):
    entry = RawStore(RAW).latest_ok(url)
    a = entry["artifact"]
    return RawArtifact.model_validate(a | {"path": Path(a["path"])})


def write_csv(name, rows):
    with open(OUT / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)

In [ ]:
raws = {r.source_url: r for r in year_artifacts(RAW, range(1981, 2019))}
season, cells, missing = {}, set(), 0
for raw in raws.values():
    year = int(raw.source_url.rsplit("=", 1)[1])
    if year not in BASE and year != 2018:
        continue
    table = S08("kl", [year]).parse(raw, kl.source("S08"))
    cells |= set(table.column("cell_id").to_pylist())
    means = daily_box_mean(table)
    missing += table.num_rows - sum(n for _, n in means.values())
    season[year] = np.array([means[d.replace(year=year)][0] for d in DAYS])
base = np.array([season[y] for y in BASE])  # 30 years x 122 days
cum = base.cumsum(axis=1)
rng = np.random.default_rng(142)  # bootstrap over years for the climatology mean
boot = cum[rng.integers(0, len(BASE), (2000, len(BASE)))].mean(axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
obs = season[2018]
total, clim = float(obs.sum()), float(cum[:, -1].mean())
print(len(cells), "cells;", missing, "missing cell-days;", round(total), "mm vs", round(clim))

In [ ]:
months = []
for m in (6, 7, 8, 9):
    idx = [i for i, d in enumerate(DAYS) if d.month == m]
    per_year = base[:, idx].sum(axis=1)
    b = per_year[rng.integers(0, len(BASE), (2000, len(BASE)))].mean(axis=1)
    o, c = float(obs[idx].sum()), float(per_year.mean())
    months.append(
        {
            "month": f"2018-{m:02d}",
            "rain_2018_mm": round(o, 1),
            "clim_1981_2010_mm": round(c, 1),
            "clim_ci95_lo_mm": round(float(np.percentile(b, 2.5)), 1),
            "clim_ci95_hi_mm": round(float(np.percentile(b, 97.5)), 1),
            "departure_pct": round((o / c - 1) * 100, 1),
        }
    )
write_csv("monthly_2018.csv", months)
for row in months:
    print(row)

In [ ]:
s08_2018 = next(r for r in raws.values() if r.source_url.endswith("=2018"))
s08_vintage = f"IMD 0.25 deg grid, 2018 file fetched {s08_2018.fetched_at:%Y-%m-%d}"
records = [
    {
        "id": f"S08-{d.isoformat()}",
        "source_id": "S08",
        "source": "IMD Pune 0.25 deg gridded rainfall, Kerala cell box mean (S08)",
        "vintage": s08_vintage,
        "kind": "daily_rain",
        "district": None,
        "date": d.isoformat(),
        "period_start": None,
        "period_end": None,
        "value": round(float(v), 2),
        "unit": "mm",
        "climatology": round(float(c), 2),
        "known_from": None,  # gridded analysis: no dated issue before the 2026 fetch
        "mode": mode(None, END),
    }
    for d, v, c in zip(DAYS, obs, base.mean(axis=0), strict=True)
]
submitted = {(t.document, t.page): t.submitted for t in TABLES}
s36_rows = []
for url in dict.fromkeys(t.url for t in TABLES):
    s36_rows += S36().parse(stored(url), kl.source("S36")).to_pylist()
for r in s36_rows:
    known = submitted[r["document_id"], r["page"]]
    records.append(
        {
            "id": f"S36-{r['document_id']}-p{r['page']}-{r['district_id']}",
            "source_id": "S36",
            "source": "KSDMA Kerala Floods 2018 memorandum (S36)",
            "vintage": f"{r['document_id']}, submitted {known:%d %b %Y}, page {r['page']}",
            "kind": r["claim_type"],
            "district": r["district_id"],
            "date": r["event_date"].isoformat() if r["event_date"] else None,
            "period_start": r["period_start"].isoformat(),
            "period_end": r["period_end"].isoformat(),
            "value": r["value"],
            "unit": r["unit"],
            "climatology": None,
            "known_from": known.isoformat(),
            "mode": mode(known, END),
        }
    )
s32_raw = stored(f"{S32_URL}?date=2018-08-15")
s32_rows = S32(kl.source("S32")).parse(s32_raw, kl.source("S32")).num_rows
replay = {
    "schema_version": 1,
    "title": "Kerala 2018 floods: descriptive replay",
    "label": LABEL,
    "window": {"start": START.isoformat(), "end": END.isoformat()},
    "forecast_vintages": [],
    "records": records,
}
validate(replay)
text = json.dumps(replay, indent=1, ensure_ascii=False) + "\n"
(OUT / "replay2018.json").write_text(text, "utf-8", newline="\n")
by_mode = {m: sum(r["mode"] == m for r in records) for m in ("as_known", "reconstructed")}
print(by_mode, "S32 dam rows on 15 Aug 2018:", s32_rows)

In [ ]:
cap = caption(
    f"S08 IMD 0.25 deg grid (reconstruction; {s08_vintage}), Kerala cell box {BOXES['kl']}",
    "daily and cumulative box-mean rain, 2018 vs 1981-2010 mean with 95% bootstrap CI",
    "mm",
    START.isoformat(),
    END.isoformat(),
    f"{len(cells)} cells, {missing} missing cell-days",
)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 7), sharex=True)
fig.subplots_adjust(bottom=0.14, hspace=0.15)
a1.bar(DAYS, obs, width=1.0, label="2018 (reconstructed)")
a1.plot(DAYS, base.mean(axis=0), color="k", lw=0.8, label="1981-2010 daily mean")
a1.set_ylabel("Daily rain (mm)")
a1.legend(fontsize=7)
a1.set_title("Kerala box rain, Jun-Sep 2018 (descriptive replay, not a forecast)", fontsize=10)
a2.fill_between(DAYS, lo, hi, color="0.8", label="1981-2010 mean, 95% bootstrap CI")
a2.plot(DAYS, cum.mean(axis=0), color="k", lw=0.8)
a2.plot(DAYS, obs.cumsum(), lw=1.5, label="2018 (reconstructed)")
a2.set_ylabel("Cumulative rain (mm)")
a2.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
a2.legend(fontsize=7)
fig.text(0.01, 0.005, textwrap.fill(cap, 110), fontsize=5)
fig.savefig(OUT / "rain_2018_vs_climatology.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
summary = {
    "rain": {
        "box": BOXES["kl"],
        "cells": len(cells),
        "missing_cell_days": int(missing),
        "jun_sep_2018_mm": round(total, 1),
        "jun_sep_clim_1981_2010_mm": round(clim, 1),
        "clim_ci95_mm": [round(float(lo[-1]), 1), round(float(hi[-1]), 1)],
        "departure_pct": round((total / clim - 1) * 100, 1),
        "first_fetch": min(r.fetched_at for r in raws.values()).isoformat(),
        "last_fetch": max(r.fetched_at for r in raws.values()).isoformat(),
    },
    "s36_rows": len(s36_rows),
    "s32": {"report": "2018-08-15", "dam_rows": s32_rows},
    "records_by_mode": by_mode,
    "forecast_vintages": 0,
    "charts": [
        {
            "file": "rain_2018_vs_climatology.png",
            "kind": "event",
            "ci": "95% bootstrap over 1981-2010 years (2000 resamples, seed 142)",
            "source": "S08 IMD 0.25 deg gridded rainfall",
            "vintage": s08_vintage,
        }
    ],
    "caption": cap,
    "limits": [
        "Descriptive only: no forecast, no model and no accuracy claim (D29, D125).",
        "No historical forecast vintage for 2018 is held; none is reconstructed or imitated.",
        "S08 is a later gridded reconstruction and is never shown in the as-known view.",
        "The cell box includes Tamil Nadu and Karnataka border cells; it is not Kerala's boundary.",
        "S36 fatalities are period totals; they are not spread over days.",
        "S36 as-known dates are the memoranda submission dates; the web copies sit under a 2019 "
        "upload path, so as-known means issued by the state then, not online then.",
        "S32 holds no 2018 dam row: the 15 Aug 2018 report is empty.",
    ],
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", "utf-8", newline="\n")
print(json.dumps(summary, indent=2))